# 03 — Ablation Exp 1–6 (proposal 3.3.3) → RQ1

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # so `import src...` works from notebooks/

import numpy as np
import pandas as pd

from src import config
from src.utils import set_seed, save_fig
set_seed()  # fix all RNGs -- reproducibility

Six conditions, identical classifier + protocol. TF-IDF sparse + dense → `sparse.hstack` in `features.assemble`. Add LinearSVM (all), MultinomialNB (TF-IDF conditions only — non-negative).

In [ ]:
from src import data, features, modeling
clean = data.load_or_build_clean(); splits = data.load_or_build_splits(clean)
y = clean[config.LABEL_COL].values

In [ ]:
# Build every block ONCE, then assemble per experiment.
blocks = {
    'tfidf':       None,  # features.build_tfidf(...)
    'stylometric': None,  # scaled
    'biber':       None,  # scaled
    'sbert':       None,  # features.build_sbert(...)
    'length':      clean[['log_token_count']].values,
}

In [ ]:
results = {}
for exp_name, which in features.EXPERIMENTS.items():
    X = features.assemble(blocks, which)
    Xtr, Xval = X[splits['train']], X[splits['val']]
    ytr, yval = y[splits['train']], y[splits['val']]
    results[exp_name] = modeling.train_and_evaluate('logreg', Xtr, ytr, Xval, yval)

pd.DataFrame({k: {'macro_f1': v.macro_f1, 'acc': v.accuracy}
              for k, v in results.items()}).T

Save the results table for the report.

In [ ]:
# TODO: dump the metrics table to figures/ or artifacts/ as csv